In [2]:
### API key management

### Reminder: Place .env file inside the root of the project folder so when calling the below from inside the notebook it should find the .env fule and load it inside the notebook environment
### PLEASE ADD THIS `.env` FILE TO YOUR PROJECT'S `.gitignore` file before committing and pushing the changes to your remote repo, as it contains API Keys and Secrets in it

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY" in os.environ)
print("LANGCHAIN_API_KEY" in os.environ)
print("TAVILY_API_KEY" in os.environ)


True
True
True


In [15]:
#magic France test cell (copy-pasted from Advanced_Retrieval_with_LangChain_Assignment.ipynb)
from langchain_openai import ChatOpenAI
from langchain_core.tracers import LangChainTracer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define components
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
prompt = ChatPromptTemplate.from_template("What is the capital of Spain?")
chain = prompt | llm | StrOutputParser()

# Attach a tracer manually (failsafe if env vars don't take)
tracer = LangChainTracer()
chain_with_tracing = chain.with_config({"callbacks": [tracer]})

# Invoke chain
response = chain_with_tracing.invoke({})
print(response)

print("\n" + "="*50)
print("🔧 RETRIEVER TEST IN MAGIC CELL CONTEXT")
print("="*50)

# Import the retriever components (you'll need to bring these over from main notebook)
# For now, let's just test if we can access your existing variables

try:
    # Test 1: Simple question to make sure LLM still works
    simple_test = llm.invoke("Test retriever context")
    print("✅ LLM still works in this context")
    
    # Test 2: Try to create a minimal retriever
    print("🧪 Creating minimal retriever test...")
    
    # You'll need to bring these from your main notebook:
    # 1. vectorstore (or recreate it)
    # 2. Your retriever chains
    # 3. Your rag_prompt
    
    # For now, let's see what variables are available
    print("📋 Variables we need to import from main notebook:")
    print("- vectorstore")
    print("- naive_retrieval_chain") 
    print("- bm25_retrieval_chain")
    print("- contextual_compression_retrieval_chain")
    print("- multi_query_retrieval_chain")
    print("- parent_document_retrieval_chain")
    print("- ensemble_retrieval_chain")
    
    print("\n🔗 This should still show up in LangSmith!")
    
except Exception as e:
    print(f"❌ Error: {e}")

    print("\n" + "="*50)
print("🚀 IMPORTING RETRIEVER COMPONENTS")
print("="*50)

# Import the components we need for retrieval
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Recreate your loan data and vectorstore
print("📄 Loading loan complaint data...")
loader = CSVLoader(
    file_path="./data/complaints.csv",
    metadata_columns=[
        "Date received", "Product", "Sub-product", "Issue", "Sub-issue", 
        "Consumer complaint narrative", "Company public response", "Company", 
        "State", "ZIP code", "Tags", "Consumer consent provided?", 
        "Submitted via", "Date sent to company", "Company response to consumer", 
        "Timely response?", "Consumer disputed?", "Complaint ID"
    ]
)

loan_complaint_data = loader.load()
for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

print(f"✅ Loaded {len(loan_complaint_data)} complaint documents")

# Recreate vectorstore with embeddings
print("🔍 Creating vectorstore...")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

print("✅ Vectorstore created")
print("🔗 This should show up in LangSmith too!")

print("\n" + "="*50)  
print("🔗 CREATING RETRIEVER CHAINS")
print("="*50)

# Create the prompt template
RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)
print("✅ Prompt template created")

# Create naive retriever
naive_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
print("✅ Naive retriever created")

# Create naive retrieval chain  
naive_retrieval_chain = (
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)
print("✅ Naive retrieval chain created")

# TEST THE CHAIN!
print("\n🧪 TESTING NAIVE RETRIEVAL CHAIN...")
question = "What is the most common issue with loans?"
result = naive_retrieval_chain.invoke({"question": question})
print("✅ Naive chain test completed!")
print(f"📝 Response: {result['response'].content[:100]}...")
print("\n🔗 CHECK LANGSMITH - This should be the first working retriever trace!")

print("\n" + "="*60)
print("🚀 FINAL LATENCY TEST - FORGET LANGSMITH")  
print("="*60)

import time

# Simple latency test for your analysis
test_question = "What is the most common issue with loans?"
retrievers_to_test = {
    "naive": naive_retrieval_chain,
}

latency_results = {}

for name, chain in retrievers_to_test.items():
    print(f"\n⏱️  Testing {name}...")
    start = time.time()
    try:
        result = chain.invoke({"question": test_question})
        latency = time.time() - start
        latency_results[name] = latency
        print(f"✅ {name}: {latency:.2f}s")
    except Exception as e:
        print(f"❌ {name}: {e}")

print(f"\n📊 LATENCY RESULTS:")
for name, latency in latency_results.items():
    print(f"  {name}: {latency:.2f}s")

print("\n🎯 YOU NOW HAVE: Performance (Ragas) + Latency!")
print("Estimate costs manually or skip for now. You can complete your analysis!")


Madrid

🔧 RETRIEVER TEST IN MAGIC CELL CONTEXT
✅ LLM still works in this context
🧪 Creating minimal retriever test...
📋 Variables we need to import from main notebook:
- vectorstore
- naive_retrieval_chain
- bm25_retrieval_chain
- contextual_compression_retrieval_chain
- multi_query_retrieval_chain
- parent_document_retrieval_chain
- ensemble_retrieval_chain

🔗 This should still show up in LangSmith!
🚀 IMPORTING RETRIEVER COMPONENTS
📄 Loading loan complaint data...
✅ Loaded 825 complaint documents
🔍 Creating vectorstore...
✅ Vectorstore created
🔗 This should show up in LangSmith too!

🔗 CREATING RETRIEVER CHAINS
✅ Prompt template created
✅ Naive retriever created
✅ Naive retrieval chain created

🧪 TESTING NAIVE RETRIEVAL CHAIN...
✅ Naive chain test completed!
📝 Response: The most common issue with loans seems to be dealing with your lender or servicer, which includes pr...

🔗 CHECK LANGSMITH - This should be the first working retriever trace!

🚀 FINAL LATENCY TEST - FORGET LANGSMITH

⏱

saving some bits that didn't work out

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=600)

scores = []

for name_eval_dataset in [bm25_eval_dataset, naive_eval_dataset, multi_query_eval_dataset, parent_doc_eval_dataset, ensemble_eval_dataset, contextual_compression_eval_dataset]:
    result = evaluate(
        dataset=name_eval_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    scores.append(result)
bm25_scores, naive_scores, multi_query_scores, parent_doc_scores, ensemble_scores, contextual_compression_scores = scores

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tracers import LangChainTracer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define components
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
prompt = ChatPromptTemplate.from_template("What is the most common issue with loans?")
chain = prompt | llm | StrOutputParser()

# Attach a tracer manually (failsafe if env vars don't take)
tracer = LangChainTracer()
chain_with_tracing = chain.with_config({"callbacks": [tracer]})

# Invoke chain
response = chain_with_tracing.invoke({})
print(response)


In [14]:
# Cell 3: Test if we can create a working retriever chain in scrap_paper
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Quick test: create a simple retriever chain
print("🧪 Creating retriever chain in working notebook...")

🧪 Creating retriever chain in working notebook...


In [5]:
# Cell 4: Create a simple retriever chain and test LangSmith
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Make sure tracing is on
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Create components with tracing enabled
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Created LLM and embeddings")

# Create a simple test document for retrieval
from langchain.schema import Document
test_docs = [
    Document(page_content="The most common loan issue is payment problems.", metadata={"id": 1}),
    Document(page_content="Many borrowers struggle with high interest rates.", metadata={"id": 2}),
    Document(page_content="Late fees are frequently mentioned in complaints.", metadata={"id": 3})
]

# Create a simple vectorstore
vectorstore = Qdrant.from_documents(
    test_docs,
    embeddings,
    location=":memory:",
    collection_name="TestComplaints"
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Create a simple RAG chain
prompt = ChatPromptTemplate.from_template("""
Answer based on context: {context}
Question: {question}
""")

simple_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": prompt | llm, "context": itemgetter("context")}
)

# TEST IT!
print("🧪 Testing simple retriever chain...")
result = simple_chain.invoke({"question": "What is the most common loan issue?"})
print("✅ Chain completed!")
print("🔗 Check LangSmith - this should show up!")

✅ Created LLM and embeddings
🧪 Testing simple retriever chain...
✅ Chain completed!
🔗 Check LangSmith - this should show up!


In [6]:
# Cell 5: Test each component separately to find the culprit
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"

print("🔬 Testing each component separately...")

# Test 1: Just the LLM (we know this works)
print("\n1️⃣ Testing LLM alone...")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
result1 = llm.invoke("Hello")
print("✅ LLM test done - should show in LangSmith")

# Test 2: Just the retriever (no LLM)
print("\n2️⃣ Testing retriever alone...")
try:
    docs = retriever.invoke("What is the most common loan issue?")
    print(f"✅ Retriever test done - got {len(docs)} docs - check LangSmith")
except Exception as e:
    print(f"❌ Retriever failed: {e}")

# Test 3: Just the prompt (no retriever)
print("\n3️⃣ Testing prompt alone...")
try:
    prompt_result = prompt.invoke({"context": "test context", "question": "test question"})
    print("✅ Prompt test done - check LangSmith")
except Exception as e:
    print(f"❌ Prompt failed: {e}")

# Test 4: Simple chain without retriever
print("\n4️⃣ Testing prompt + LLM (no retriever)...")
try:
    simple_prompt_chain = prompt | llm
    result4 = simple_prompt_chain.invoke({"context": "The most common issue is payment problems", "question": "What is the most common issue?"})
    print("✅ Prompt+LLM test done - check LangSmith")
except Exception as e:
    print(f"❌ Prompt+LLM failed: {e}")

print("\n🔗 Check LangSmith - which tests showed up?")

🔬 Testing each component separately...

1️⃣ Testing LLM alone...
✅ LLM test done - should show in LangSmith

2️⃣ Testing retriever alone...
✅ Retriever test done - got 2 docs - check LangSmith

3️⃣ Testing prompt alone...
✅ Prompt test done - check LangSmith

4️⃣ Testing prompt + LLM (no retriever)...
✅ Prompt+LLM test done - check LangSmith

🔗 Check LangSmith - which tests showed up?


In [12]:
# Cell 8: Clean latency test for all 6 retrievers
import time
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

print("🚀 LATENCY TEST FOR ALL 6 RETRIEVERS")
print("="*50)

# Use the components from Cell 1 (naive_retrieval_chain, vectorstore, etc.)
test_question = "What is the most common issue with loans?"

# Create the other 5 chains quickly
print("Building retriever chains...")

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data)
bm25_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)

# Add the other 4 chains here...

# Test all for latency
chains = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_chain,
    # ... add others
}

results = {}
for name, chain in chains.items():
    start = time.time()
    try:
        chain.invoke({"question": test_question})
        results[name] = time.time() - start
        print(f"✅ {name}: {results[name]:.2f}s")
    except Exception as e:
        print(f"❌ {name}: {e}")

print("\n📊 FINAL RESULTS:")
for name, latency in results.items():
    print(f"  {name}: {latency:.2f}s")

🚀 LATENCY TEST FOR ALL 6 RETRIEVERS
Building retriever chains...
✅ naive: 1.49s
✅ bm25: 0.73s

📊 FINAL RESULTS:
  naive: 1.49s
  bm25: 0.73s


In [13]:
# Cell 9: Test all 6 retrievers with golden dataset
import time
import json
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

print("🚀 TESTING ALL 6 RETRIEVERS WITH GOLDEN DATASET")
print("="*60)

# Load your golden dataset
with open("goldendataset.json", "r") as f:
    golden_data = json.load(f)

test_questions = [item["eval_sample"]["user_input"] for item in golden_data]
print(f"📝 Testing with {len(test_questions)} questions from golden dataset")

# Create all 6 retriever chains
print("🔧 Building all retriever chains...")

# 1. Naive (already exists)
# 2. BM25
bm25_retriever = BM25Retriever.from_documents(loan_complaint_data)
bm25_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)

# 3. Multi-Query (makes extra LLM calls = higher cost)
multi_query_retriever = MultiQueryRetriever.from_llm(retriever=naive_retriever, llm=llm)
multi_query_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)

# 4. Parent Document
store = InMemoryStore()
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)
parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore, docstore=store, child_splitter=child_splitter
)
parent_doc_retriever.add_documents(loan_complaint_data[:100], ids=None)
parent_doc_chain = (
    {"context": itemgetter("question") | parent_doc_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)

# 5. Ensemble
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, naive_retriever], weights=[0.5, 0.5]
)
ensemble_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)

# 6. Contextual Compression (makes extra LLM calls = higher cost)
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)
compression_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | llm, "context": itemgetter("context")}
)

print("✅ All 6 chains created!")

# Test all retrievers with full golden dataset
chains = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_chain,
    "multi_query": multi_query_chain,
    "parent_doc": parent_doc_chain,
    "ensemble": ensemble_chain,
    "contextual_compression": compression_chain,
}

results = {}
for name, chain in chains.items():
    print(f"\n🔄 Testing {name} with {len(test_questions)} questions...")
    start_time = time.time()
    question_times = []
    
    for i, question in enumerate(test_questions):
        q_start = time.time()
        try:
            chain.invoke({"question": question})
            q_time = time.time() - q_start
            question_times.append(q_time)
            if i % 3 == 0:  # Progress indicator
                print(f"  Question {i+1}/{len(test_questions)}: {q_time:.2f}s")
        except Exception as e:
            print(f"  ❌ Question {i+1} failed: {e}")
            question_times.append(None)
    
    total_time = time.time() - start_time
    avg_time = sum(t for t in question_times if t) / len([t for t in question_times if t])
    
    results[name] = {
        "total_time": total_time,
        "avg_time": avg_time,
        "question_times": question_times
    }
    
    print(f"✅ {name}: Total={total_time:.2f}s, Avg={avg_time:.2f}s")

print(f"\n📊 FINAL LATENCY RESULTS:")
print(f"{'Retriever':<20} {'Total Time':<12} {'Avg/Query':<12} {'Est. Cost'}")
print("-" * 60)

for name, data in results.items():
    # Cost estimation: Multi-query and compression make extra LLM calls
    cost_multiplier = 1
    if name == "multi_query":
        cost_multiplier = 3  # ~3x LLM calls for query generation
    elif name == "contextual_compression":
        cost_multiplier = 2  # ~2x LLM calls for compression
    
    est_cost = f"{cost_multiplier}x baseline"
    
    print(f"{name:<20} {data['total_time']:<12.2f} {data['avg_time']:<12.2f} {est_cost}")

print(f"\n🎯 YOU NOW HAVE: Performance (Ragas) + Latency + Cost estimates!")

🚀 TESTING ALL 6 RETRIEVERS WITH GOLDEN DATASET
📝 Testing with 10 questions from golden dataset
🔧 Building all retriever chains...
✅ All 6 chains created!

🔄 Testing naive with 10 questions...
  Question 1/10: 0.87s
  Question 4/10: 0.99s
  Question 7/10: 1.87s
  Question 10/10: 3.72s
✅ naive: Total=21.06s, Avg=2.11s

🔄 Testing bm25 with 10 questions...
  Question 1/10: 0.56s
  Question 4/10: 0.72s
  Question 7/10: 1.50s
  Question 10/10: 2.28s
✅ bm25: Total=14.26s, Avg=1.43s

🔄 Testing multi_query with 10 questions...
  Question 1/10: 2.91s
  Question 4/10: 2.64s
  ❌ Question 7 failed: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 18302 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
  Question 10/10: 4.24s
✅ multi_query: Total=35.41s, Avg=3.60s

🔄 Testing parent_doc with 10 questions...
  Question 1/10: 0.66s
 